In [6]:
# ============================================
# ÉTAPE 1 : Importer les bibliothèques
# ============================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import joblib
import json
import os
from datetime import datetime
from google.colab import files

print("="*60)
print("MILESTONE 2 - ENTRAÎNEMENT DU MODÈLE")
print("="*60)

# Créer les dossiers nécessaires
os.makedirs('models', exist_ok=True)
os.makedirs('src', exist_ok=True)

# ============================================
# ÉTAPE 2 : Charger le dataset
# ============================================
print("\n ÉTAPE 2 : Chargement du dataset")

# Vérifier si le dataset existe déjà
dataset_path = 'data/processed/credit_processed.csv'

if os.path.exists(dataset_path):
    print(f" Dataset trouvé : {dataset_path}")
    df = pd.read_csv(dataset_path)
else:
    print(" Veuillez uploader votre dataset ML-ready (credit_processed.csv) :")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)

    # Créer le dossier data/processed s'il n'existe pas
    os.makedirs('data/processed', exist_ok=True)

    # Sauvegarder une copie
    import shutil
    shutil.move(filename, dataset_path)
    print(f" Dataset sauvegardé : {dataset_path}")

print(f"   - Lignes : {df.shape[0]}")
print(f"   - Colonnes : {df.shape[1]}")
print("\nAperçu des données :")
display(df.head())

# ============================================
# ÉTAPE 3 : Séparer X (features) et y (cible)
# ============================================
print("\n ÉTAPE 3 : Séparation X et y")

# Identifier la colonne cible
target_column = 'credit_risk_encoded'

if target_column not in df.columns:
    print(f" Colonne '{target_column}' non trouvée !")
    print("Colonnes disponibles :", df.columns.tolist())

    # Rechercher une colonne similaire
    possible_targets = [col for col in df.columns if 'credit' in col.lower() or 'risk' in col.lower() or 'target' in col.lower()]
    if possible_targets:
        target_column = possible_targets[0]
        print(f" Colonne cible sélectionnée : {target_column}")
    else:
        raise ValueError("Impossible de trouver la colonne cible !")

# Séparer les features et la cible
X = df.drop(columns=[target_column])
y = df[target_column]

# Supprimer les colonnes non numériques si elles existent
for col in X.columns:
    if X[col].dtype == 'object':
        print(f" Colonne non numérique détectée : {col} - Suppression")
        X = X.drop(columns=[col])

print(f" X (features) : {X.shape[0]} lignes, {X.shape[1]} colonnes")
print(f" y (cible) : {y.shape[0]} lignes")
print(f"\nDistribution de la cible :")
print(y.value_counts())
print(f"  - 0 (good) : {(y == 0).sum()} clients")
print(f"  - 1 (bad) : {(y == 1).sum()} clients")

# ============================================
# ÉTAPE 4 : Séparer les données (Train/Test)
# ============================================
print("\n ÉTAPE 4 : Séparation Train/Test")

# Séparation 80% Train / 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Garder la même proportion de classes
)

print(f" Train set : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f" Test set : {X_test.shape[0]} lignes ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nDistribution dans Train :")
print(y_train.value_counts())
print(f"\nDistribution dans Test :")
print(y_test.value_counts())

# ============================================
# ÉTAPE 5 : Choisir un modèle
# ============================================
print("\n ÉTAPE 5 : Choix du modèle")

# Option 1 : Random Forest (recommandé)
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model_name = "Random Forest"

# Option 2 : Logistic Regression (alternative)
# model = LogisticRegression(random_state=42, max_iter=1000)
# model_name = "Logistic Regression"

# Option 3 : Decision Tree (alternative)
# model = DecisionTreeClassifier(random_state=42, max_depth=5)
# model_name = "Decision Tree"

print(f" Modèle choisi : {model_name}")
print(f"   Paramètres : {model.get_params()}")

# ============================================
# ÉTAPE 6 : Entraîner le modèle
# ============================================
print("\n ÉTAPE 6 : Entraînement du modèle")

print("Entraînement en cours...")
start_time = datetime.now()

model.fit(X_train, y_train)

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds()

print(f" Entraînement terminé !")
print(f"   - Temps d'entraînement : {training_time:.2f} secondes")

# ============================================
# ÉTAPE 7 : Évaluer le modèle
# ============================================
print("\n ÉTAPE 7 : Évaluation du modèle")

# Prédictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Métriques pour Test
accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, average='binary')
recall = recall_score(y_test, y_pred_test, average='binary')
f1 = f1_score(y_test, y_pred_test, average='binary')

# Métriques pour Train
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train, average='binary')
recall_train = recall_score(y_train, y_pred_train, average='binary')
f1_train = f1_score(y_train, y_pred_train, average='binary')

print("\n Performances sur le Test Set :")
print(f"   - Accuracy  : {accuracy:.4f}")
print(f"   - Precision : {precision:.4f}")
print(f"   - Recall    : {recall:.4f}")
print(f"   - F1-Score  : {f1:.4f}")

print("\n Performances sur le Train Set :")
print(f"   - Accuracy  : {accuracy_train:.4f}")
print(f"   - Precision : {precision_train:.4f}")
print(f"   - Recall    : {recall_train:.4f}")
print(f"   - F1-Score  : {f1_train:.4f}")

# Vérifier le sur-apprentissage
overfitting = accuracy_train - accuracy
if overfitting > 0.10:
    print(f"\n Attention : Sur-apprentissage possible !")
    print(f"   Différence Train/Test : {overfitting:.4f}")

# Rapport de classification détaillé
print("\n Rapport de classification détaillé :")
print(classification_report(y_test, y_pred_test, target_names=['Good', 'Bad']))

# Matrice de confusion
print("\n Matrice de confusion :")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)
print(f"\n   - Vrais Négatifs (Good prédit Good) : {cm[0][0]}")
print(f"   - Faux Positifs (Bad prédit Good)  : {cm[0][1]}")
print(f"   - Faux Négatifs (Good prédit Bad)  : {cm[1][0]}")
print(f"   - Vrais Positifs (Bad prédit Bad)  : {cm[1][1]}")

# ============================================
# ÉTAPE 8 : Sauvegarder le modèle
# ============================================
print("\n ÉTAPE 8 : Sauvegarde du modèle")

# Sauvegarder le modèle
model_path = 'models/model.pkl'
joblib.dump(model, model_path)
print(f" Modèle sauvegardé : {model_path}")
print(f"   - Taille du fichier : {os.path.getsize(model_path) / 1024:.2f} KB")

# ============================================
# ÉTAPE 9 : Sauvegarder les métriques
# ============================================
print("\n ÉTAPE 9 : Sauvegarde des métriques")

# Créer le dictionnaire des métriques
metrics = {
    "model_info": {
        "name": model_name,
        "type": str(type(model).__name__),
        "parameters": str(model.get_params())
    },
    "training": {
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "training_time_seconds": training_time,
        "train_size": int(X_train.shape[0]),
        "test_size": int(X_test.shape[0]),
        "total_samples": int(len(X)),
        "features_count": int(X.shape[1])
    },
    "performance_test": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1)
    },
    "performance_train": {
        "accuracy": float(accuracy_train),
        "precision": float(precision_train),
        "recall": float(recall_train),
        "f1_score": float(f1_train)
    },
    "confusion_matrix": {
        "true_negative": int(cm[0][0]),
        "false_positive": int(cm[0][1]),
        "false_negative": int(cm[1][0]),
        "true_positive": int(cm[1][1])
    },
    "class_distribution": {
        "class_0_good": int((y == 0).sum()),
        "class_1_bad": int((y == 1).sum())
    }
}

# Sauvegarder les métriques en JSON
metrics_path = 'models/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f" Métriques sauvegardées : {metrics_path}")

# ============================================
# RÉSUMÉ FINAL
# ============================================
print("\n" + "="*60)
print(" RÉSUMÉ FINAL - MILESTONE 2")
print("="*60)

print(f"""
 MODÈLE ENTRAÎNÉ AVEC SUCCÈS !

 Fichiers créés :
   - models/model.pkl        : Modèle entraîné
   - models/metrics.json     : Métriques de performance

 Performances du modèle (Test Set) :
   - Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)
   - Precision : {precision:.4f} ({precision*100:.2f}%)
   - Recall    : {recall:.4f} ({recall*100:.2f}%)
   - F1-Score  : {f1:.4f} ({f1*100:.2f}%)

 Performances du modèle (Train Set) :
   - Accuracy  : {accuracy_train:.4f} ({accuracy_train*100:.2f}%)
   - Precision : {precision_train:.4f} ({precision_train*100:.2f}%)
   - Recall    : {recall_train:.4f} ({recall_train*100:.2f}%)
   - F1-Score  : {f1_train:.4f} ({f1_train*100:.2f}%)

 Matrice de confusion :
   - Vrais Négatifs  : {cm[0][0]}
   - Faux Positifs   : {cm[0][1]}
   - Faux Négatifs   : {cm[1][0]}
   - Vrais Positifs  : {cm[1][1]}

 Distribution des classes :
   - 0 (good) : {(y == 0).sum()} clients
   - 1 (bad)  : {(y == 1).sum()} clients
""")

print("="*60)
print(" MILESTONE 2 - ÉTAPES 1 À 9 TERMINÉES AVEC SUCCÈS !")
print("="*60)

# ============================================
# TÉLÉCHARGEMENT DES FICHIERS (Optionnel)
# ============================================
print("\n Téléchargement des fichiers sur votre ordinateur ?")

download_choice = input("Voulez-vous télécharger model.pkl et metrics.json ? (o/n) : ")

if download_choice.lower() == 'o':
    print("\n Téléchargement de model.pkl...")
    files.download('models/model.pkl')

    print("\n Téléchargement de metrics.json...")
    files.download('models/metrics.json')

    print("\n Tous les fichiers ont été téléchargés !")
else:
    print("\n Les fichiers sont sauvegardés localement dans le dossier 'models/'")

MILESTONE 2 - ENTRAÎNEMENT DU MODÈLE

 ÉTAPE 2 : Chargement du dataset
 Dataset trouvé : data/processed/credit_processed.csv
   - Lignes : 3200
   - Colonnes : 41

Aperçu des données :


,age_group,gender_encoded,marital_status_encoded,housing_status_encoded,income_category,customer_reference,annual_income_normalized,savings_balance_normalized,checking_balance_normalized,loan_amount_normalized,...,region_Trarza,education_level_master,education_level_phd,education_level_primary,education_level_secondary,employment_status_public_sector,employment_status_retired,employment_status_self_employed,employment_status_student,employment_status_unemployed
0,36-45,1,2,2,Élevé,USER0001,0.174551,1.040954,0.131306,0.630591,...,False,False,False,False,False,True,False,False,False,False
1,36-45,1,1,3,Élevé,USER0002,0.429863,0.081151,-1.279019,1.487241,...,False,False,False,False,True,False,False,True,False,False
2,26-35,0,1,3,Élevé,USER0003,0.418044,1.321549,-0.978143,-1.461699,...,False,False,False,False,False,False,False,True,False,False
3,46-55,0,0,3,Très bas,USER0004,-0.984885,-0.266905,0.048140,0.024276,...,False,False,False,True,False,False,False,True,False,False
4,46-55,1,1,2,Très bas,USER0005,-1.421001,-0.804513,-0.667361,-0.494273,...,False,False,False,False,True,False,False,False,True,False



 ÉTAPE 3 : Séparation X et y
 Colonne non numérique détectée : age_group - Suppression
 Colonne non numérique détectée : income_category - Suppression
 Colonne non numérique détectée : customer_reference - Suppression
 X (features) : 3200 lignes, 37 colonnes
 y (cible) : 3200 lignes

Distribution de la cible :
credit_risk_encoded
0    1744
1    1456
Name: count, dtype: int64
  - 0 (good) : 1744 clients
  - 1 (bad) : 1456 clients

 ÉTAPE 4 : Séparation Train/Test
 Train set : 2560 lignes (80.0%)
 Test set : 640 lignes (20.0%)

Distribution dans Train :
credit_risk_encoded
0    1395
1    1165
Name: count, dtype: int64

Distribution dans Test :
credit_risk_encoded
0    349
1    291
Name: count, dtype: int64

 ÉTAPE 5 : Choix du modèle
 Modèle choisi : Random Forest
   Paramètres : {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_